In [1]:
import pandas as pd
import nltk
import string

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

import gensim
from gensim import corpora
from gensim.models import CoherenceModel

In [2]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     /home/4e4d763e-33da-474c-88ed-
[nltk_data]     bac8348e6ba4/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/4e4d763e-33da-474c-88ed-
[nltk_data]     bac8348e6ba4/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/4e4d763e-33da-474c-88ed-
[nltk_data]     bac8348e6ba4/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
df = pd.read_csv('news_dataset.csv', engine='python', on_bad_lines='skip')

print("COLUMNS:", df.columns)

texts = df.apply(lambda col: col.astype(str).str.len().mean()).idxmax()

print("Selected column:", texts)

texts = df[texts]
texts = texts.dropna().astype(str)

print("Total data:", len(texts))
print(texts.head())

COLUMNS: Index(['Unnamed: 0', 'text', 'target', 'title', 'date'], dtype='object')
Selected column: text
Total data: 11096
0    I was wondering if anyone out there could enli...
1    I recently posted an article asking what kind ...
2    \nIt depends on your priorities.  A lot of peo...
3    an excellent automatic can be found in the sub...
4    : Ford and his automobile.  I need information...
Name: text, dtype: object


In [4]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    tokens = word_tokenize(text)
    
    tokens = [lemmatizer.lemmatize(word) 
              for word in tokens 
              if word not in stop_words]
    
    return tokens

processed_texts = texts.apply(preprocess)

processed_texts = processed_texts[processed_texts.map(len) > 0]

print("After cleaning:", len(processed_texts))
print(processed_texts.head())

After cleaning: 11000
0    [wondering, anyone, could, enlighten, car, saw...
1    [recently, posted, article, asking, kind, rate...
2    [depends, priority, lot, people, put, higher, ...
3    [excellent, automatic, found, subaru, legacy, ...
4    [ford, automobile, need, information, whether,...
Name: text, dtype: object


In [5]:
dictionary = corpora.Dictionary(processed_texts)

corpus = [dictionary.doc2bow(text) for text in processed_texts]

print("Corpus size:", len(corpus))

Corpus size: 11000


In [ ]:
lda_model = gensim.models.LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=4,
    random_state=42,
    passes=10
)

In [7]:
print("\n===== TOPICS =====")
for i, topic in lda_model.print_topics(num_words=5):
    print(f"Topic {i}: {topic}")


===== TOPICS =====
Topic 0: 0.009*"image" + 0.008*"data" + 0.007*"available" + 0.006*"file" + 0.006*"system"
Topic 1: 0.007*"space" + 0.006*"car" + 0.005*"would" + 0.004*"like" + 0.004*"one"
Topic 2: 0.011*"space" + 0.004*"would" + 0.004*"3" + 0.004*"like" + 0.004*"satellite"
Topic 3: 0.010*"probe" + 0.008*"mission" + 0.008*"space" + 0.007*"first" + 0.007*"would"


The LDA model successfully generated four topics from the dataset.

Topic 0 is related to data processing and digital systems, as indicated by keywords such as "image", "data", and "system".

Topic 1 appears to represent a general or mixed discussion, possibly involving transportation and general context.

Topic 2 focuses on space technology, particularly satellite-related content.

Topic 3 highlights space exploration themes, including missions and probes.

The coherence score further supports that the generated topics are meaningful and interpretable.

In [ ]:
coherence_model = CoherenceModel(
    model=lda_model,
    texts=processed_texts,
    dictionary=dictionary,
    coherence='c_v'
)

score = coherence_model.get_coherence()
print("\nCoherence Score:", score)

The coherence score obtained is 0.4048, which indicates a moderate level of topic coherence.

This suggests that the words within each topic are reasonably related and semantically meaningful.

Although the score is not extremely high, it is acceptable for topic modeling tasks and demonstrates that the LDA model has successfully identified interpretable themes within the dataset.

Overall, the model is effective in extracting useful insights from the text data.